# Character Identity Agent 테스트 (Production Level)

캐릭터 기본 정보 추출 에이전트 테스트 노트북

## 역할: "Identity Analyst" (신원 분석가)
- 이름, 나이, 성별
- **직업/클래스 (occupation)** ✨ NEW
- 역할 (protagonist/antagonist/supporting)
- 소속 (faction)
- 배경 스토리 (backstory) - Fallback 정책 적용

> **핵심 검증**: 언어 일관성 + occupation/backstory 추출

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
SAMPLE_STORY = """아린은 어두운 숲 한가운데 서 있었다. 스물다섯 살의 젊은 여전사는 긴 검은 머리카락을 바람에 휘날리며, 손에 쥔 은빛 검을 꼭 움켜쥐었다.

그림자 속에서 카엘이 나타났다. 서른 살의 전직 기사는 검은 갑옷을 입고 있었고, 얼굴에는 오래된 흉터가 새겨져 있었다. '오랜만이군, 아린.' 카엘의 목소리는 차가웠다.

두 사람은 한때 '은빛 여명' 기사단의 동맹이었다. 하지만 5년 전 대전쟁 이후, 카엘은 왕국을 배신하고 '암흑회'에 가담했다."""

def create_base_state(story=SAMPLE_STORY):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Identity Agent 실행

In [3]:
from app.agents.extraction.character.identity import identity_extraction_node

async def test_identity():
    print("🆔 Identity Agent 테스트...")
    return await identity_extraction_node(create_base_state())

result = run_async(test_identity())

# 에러 확인
if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    identity_data = result.get('char_identity', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(identity_data)}개")
    print(f"   - 이름: {list(identity_data.keys())}")

🆔 Identity Agent 테스트...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['아린', '카엘']


## 2. Human-Readable 출력

In [4]:
identity_data = result.get('char_identity', {})

if not identity_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🆔 Identity Data (캐릭터별 기본 정보)")
    print("="*70)

    for name, data in identity_data.items():
        print(f"\n🧑 {name}")
        print(f"   나이: {data.get('age', 'N/A')}")
        print(f"   성별: {data.get('gender', 'N/A')}")
        print(f"   종족: {data.get('race', 'N/A')}")
        print(f"   ⚔️ 직업: {data.get('occupation', 'N/A')}")
        print(f"   역할: {data.get('role', 'N/A')}")
        print(f"   소속: {data.get('faction', 'N/A')}")
        print(f"   별칭: {data.get('aliases', [])}")
        print(f"   상태: {data.get('status', 'N/A')}")
        backstory = data.get('backstory', '')
        if backstory:
            if len(backstory) > 50:
                print(f"   📜 배경: {backstory[:50]}...")
            else:
                print(f"   📜 배경: {backstory}")
        else:
            print(f"   📜 배경: (없음)")

🆔 Identity Data (캐릭터별 기본 정보)

🧑 아린
   나이: 25
   성별: female
   종족: None
   ⚔️ 직업: 여전사
   역할: protagonist
   소속: 은빛 여명 기사단
   별칭: []
   상태: alive
   📜 배경: 젊은 여전사, 은빛 여명 기사단 소속

🧑 카엘
   나이: 30
   성별: male
   종족: None
   ⚔️ 직업: 전직 기사
   역할: antagonist
   소속: 암흑회
   별칭: []
   상태: alive
   📜 배경: 전직 기사, 5년 전 왕국을 배신하고 암흑회에 가담


## 3. ⚔️ Occupation 및 Backstory 검증 (NEW)

In [5]:
print("="*70)
print("⚔️ Occupation & Backstory 검증 (NEW)")
print("="*70)

identity_data = result.get('char_identity', {})

EXPECTED_DATA = {
    '아린': {
        'occupation_keywords': ['여전사', '전사', 'warrior'],
        'backstory_keywords': ['여전사', '은빛 여명', '기사단']
    },
    '카엘': {
        'occupation_keywords': ['기사', '전직 기사', 'knight'],
        'backstory_keywords': ['전직 기사', '배신', '암흑회', '대전쟁']
    }
}

for name, expected in EXPECTED_DATA.items():
    data = identity_data.get(name, {})
    
    print(f"\n{name}:")
    
    # Occupation 검증
    occupation = str(data.get('occupation', '') or '').lower()
    if any(kw.lower() in occupation for kw in expected['occupation_keywords']):
        print(f"   ✅ occupation: {data.get('occupation')}")
    elif occupation:
        print(f"   ⚠️ occupation: {data.get('occupation')} (예상 키워드: {expected['occupation_keywords']})")
    else:
        print(f"   ❌ occupation: (없음) - 예상: {expected['occupation_keywords']}")
    
    # Backstory 검증
    backstory = str(data.get('backstory', '') or '').lower()
    if any(kw.lower() in backstory for kw in expected['backstory_keywords']):
        print(f"   ✅ backstory: {data.get('backstory')[:40]}..." if len(backstory) > 40 else f"   ✅ backstory: {data.get('backstory')}")
    elif backstory:
        print(f"   ⚠️ backstory 내용 부족: {data.get('backstory')}")
    else:
        print(f"   ❌ backstory: (없음) - Fallback 정책 미작동")

⚔️ Occupation & Backstory 검증 (NEW)

아린:
   ✅ occupation: 여전사
   ✅ backstory: 젊은 여전사, 은빛 여명 기사단 소속

카엘:
   ✅ occupation: 전직 기사
   ✅ backstory: 전직 기사, 5년 전 왕국을 배신하고 암흑회에 가담


## 4. 언어 일관성 검증

> **핵심**: 한국어 입력 → 한국어 출력 (번역 안 됨)

In [6]:
print("="*70)
print("🌐 언어 일관성 검증")
print("="*70)

identity_data = result.get('char_identity', {})

# 아린 검증
arin = identity_data.get('아린', {})
if arin:
    faction = arin.get('faction', '')
    if faction and ('은빛' in faction or '여명' in faction or '기사단' in faction):
        print(f"✅ 아린 faction 한국어 유지: {faction}")
    elif faction:
        print(f"⚠️ 아린 faction 번역됨?: {faction}")
    else:
        print("⚠️ 아린 faction 없음")
else:
    print("❌ 아린 not found")

# 카엘 검증
kael = identity_data.get('카엘', {})
if kael:
    faction = kael.get('faction', '')
    if faction and '암흑회' in faction:
        print(f"✅ 카엘 faction 한국어 유지: {faction}")
    elif faction:
        print(f"⚠️ 카엘 faction 번역됨?: {faction}")
    else:
        print("⚠️ 카엘 faction 없음")
else:
    print("❌ 카엘 not found")

🌐 언어 일관성 검증
✅ 아린 faction 한국어 유지: 은빛 여명 기사단
✅ 카엘 faction 한국어 유지: 암흑회


## 5. Full JSON 출력

In [7]:
print("="*70)
print("📄 Full JSON Output")
print("="*70)
identity_data = result.get('char_identity', {})
if identity_data:
    print(json.dumps(identity_data, ensure_ascii=False, indent=2))
else:
    print("{}")
    print("\n❌ 데이터 없음")

📄 Full JSON Output
{
  "아린": {
    "name": "아린",
    "age": 25,
    "gender": "female",
    "race": null,
    "occupation": "여전사",
    "faction": "은빛 여명 기사단",
    "role": "protagonist",
    "aliases": [],
    "status": "alive",
    "backstory": "젊은 여전사, 은빛 여명 기사단 소속"
  },
  "카엘": {
    "name": "카엘",
    "age": 30,
    "gender": "male",
    "race": null,
    "occupation": "전직 기사",
    "faction": "암흑회",
    "role": "antagonist",
    "aliases": [],
    "status": "alive",
    "backstory": "전직 기사, 5년 전 왕국을 배신하고 암흑회에 가담"
  }
}


## 6. Production 체크리스트

In [8]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

identity_data = result.get('char_identity', {})
checks = []

# 1. 캐릭터 존재
if len(identity_data) >= 2:
    checks.append(("✅", "2+ characters extracted"))
elif len(identity_data) == 1:
    checks.append(("⚠️", f"Only 1 character (expected 2)"))
else:
    checks.append(("❌", f"Only {len(identity_data)} characters"))

if identity_data:
    # 2. Occupation 추출 (NEW)
    has_occupation = any(data.get('occupation') for data in identity_data.values())
    if has_occupation:
        checks.append(("✅", "Occupation extracted for some characters"))
    else:
        checks.append(("❌", "Missing occupation (should extract 여전사, 기사 etc.)"))
    
    # 3. Backstory 추출 (Enhanced)
    has_backstory = all(data.get('backstory') for data in identity_data.values())
    if has_backstory:
        checks.append(("✅", "Backstory: All characters have backstory (Fallback OK)"))
    elif any(data.get('backstory') for data in identity_data.values()):
        checks.append(("⚠️", "Backstory: Some characters missing (Fallback 개선 필요)"))
    else:
        checks.append(("❌", "Backstory: No backstory extracted"))
    
    # 4. 이름/역할 추출
    has_names = all(data.get('name') for data in identity_data.values())
    has_roles = all(data.get('role') for data in identity_data.values())
    if has_names and has_roles:
        checks.append(("✅", "All characters have names and roles"))
    else:
        checks.append(("❌", "Missing names or roles"))
    
    # 5. 언어 일관성
    korean_names = all(any(ord(c) >= 0xAC00 and ord(c) <= 0xD7A3 for c in name) for name in identity_data.keys())
    if korean_names:
        checks.append(("✅", "Language consistency: Korean names preserved"))
    else:
        checks.append(("⚠️", "Language consistency: Names may be translated"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
total = len(checks)
print(f"결과: {passed}/{total} checks passed")

✅ Production 체크리스트

✅ 2+ characters extracted
✅ Occupation extracted for some characters
✅ Backstory: All characters have backstory (Fallback OK)
✅ All characters have names and roles
✅ Language consistency: Korean names preserved

결과: 5/5 checks passed


## 7. 디버그 정보

In [9]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"\nResult keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Messages: {result.get('messages', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")

🔍 디버그 정보

Result keys: dict_keys(['char_identity', 'completed_agents', 'messages'])
Errors: []
Messages: [{'role': 'identity_agent', 'content': 'Extracted 2 character identities'}]
Completed agents: ['identity']
